# MLP Gradient Flow

How do gradients flow through MLP layers? Weight gradient norms,
per-neuron gradient profiles, and gradient sparsity.

## Why This Matters

MLP gradient flow provides tools for understanding how feed-forward layers transform and store information. Since MLPs have been shown to function as key-value memories, understanding their internal mechanics is essential for locating knowledge and understanding factual recall.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Bills et al. (2023) "Language Models Can Explain Neurons"](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) — Automated neuron interpretation methods

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_gradient_flow import (
    mlp_input_gradient, mlp_weight_gradient_norms,
    mlp_neuron_gradient_profile, mlp_gradient_sparsity,
    mlp_gradient_flow_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## MLP Input Gradient

Gradient magnitude at MLP inputs shows which positions contribute most.

In [ ]:
for layer in range(4):
    result = mlp_input_gradient(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_grad={result['mean_grad_norm']:.6f}")
    for p in result['per_position']:
        bar = '█' * int(p['grad_norm'] * 100)
        print(f"    Pos {p['position']}: {p['grad_norm']:.6f} {bar}")

## Weight Gradient Norms

How large are gradients for each MLP weight matrix?

In [ ]:
for layer in range(4):
    result = mlp_weight_gradient_norms(model, tokens, layer=layer)
    print(f"  Layer {layer}: W_in={result['W_in_grad_norm']:.6f}, "
          f"W_out={result['W_out_grad_norm']:.6f}, "
          f"total={result['total_grad_norm']:.6f}")

## Neuron Gradient Profile

Which neurons receive the largest gradient signal?

In [ ]:
for layer in range(4):
    result = mlp_neuron_gradient_profile(model, tokens, layer=layer, top_k=5)
    print(f"  Layer {layer}: mean={result['mean_neuron_grad']:.6f}, max={result['max_neuron_grad']:.6f}")
    for n in result['top_neurons'][:3]:
        print(f"    N{n['neuron']}: grad={n['grad_norm']:.6f}")

## Gradient Sparsity

What fraction of neurons receive significant gradient signal?

In [ ]:
for layer in range(4):
    result = mlp_gradient_sparsity(model, tokens, layer=layer)
    print(f"  Layer {layer}: sparsity={result['gradient_sparsity']:.3f}, "
          f"{result['n_significant']}/{result['total_neurons']} significant")

## Gradient Flow Summary

In [ ]:
result = mlp_gradient_flow_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: total_grad={p['total_grad_norm']:.6f}, "
          f"sparsity={p['gradient_sparsity']:.3f}")